In [11]:
import numpy as np
import pandas as pd
import dynamo as dyn
from dynamo.preprocessing import Preprocessor

In [12]:
%%time
# Load processed adata

fpath = "/nfs/turbo/umms-indikar/shared/projects/cell_cycle/pipeline_outputs/single_cell_fibroblast/scanpy/processed.anndata.h5ad"

adata = dyn.read_h5ad(fpath)
adata

CPU times: user 194 ms, sys: 1.57 s, total: 1.76 s
Wall time: 9.09 s


AnnData object with n_obs × n_vars = 7748 × 14082
    obs: 'n_genes', 'cell', 'G1', 'G2M', 'S', 'pred_phase', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'leiden', 'pred_G1', 'pred_S', 'pred_G2M', 'dpt_pseudotime'
    var: 'gene_name', 'Chromosome', 'Start', 'End', 'Strand', 'seurat_S', 'seurat_G2M', 'is_seurat', 'is_kegg', 'whitfield_G1/S', 'whitfield_G2', 'whitfield_G2/M', 'whitfield_M/G1', 'whitfield_S', 'is_whitfield', 'GO_G1', 'GO_G1/S', 'GO_G2', 'GO_G2/M', 'GO_M', 'GO_S', 'is_GO', 'LIU_G2MvsG0G1', 'LIU_G2MvsS', 'LIU_SvsG0G1', 'is_LIU', 'cell_cycle', 'n_cells', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_count

In [13]:
adata.obs.head()

,n_genes,cell,G1,G2M,S,pred_phase,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,...,log1p_total_counts_ribo,pct_counts_ribo,total_counts_hb,log1p_total_counts_hb,pct_counts_hb,leiden,pred_G1,pred_S,pred_G2M,dpt_pseudotime
cell_id,,,,,,,,,,,,,,,,,,,,,
AAACCAAAGGGTAGCA,3701,AAACCAAAGGGTAGCA,80.0,403.0,87.0,G2M,3701,8.216628,16001,9.680469,...,3.871201,0.293732,0,0.000000,0.000000,0,4.394449,4.477337,6.001415,0.391558
AAACCAAAGTAAGGGT,2280,AAACCAAAGTAAGGGT,18.0,5.0,7.0,G1,2280,7.732369,7864,8.970178,...,2.708050,0.178026,0,0.000000,0.000000,1,2.944439,2.079442,1.791759,0.000000
AAACCATTCAGGTAGG,3715,AAACCATTCAGGTAGG,43.0,183.0,59.0,G2M,3714,8.220134,15609,9.655667,...,3.912023,0.313921,1,0.693147,0.006407,0,3.784190,4.094345,5.214936,0.382223
AAACCATTCCAGCCCT,2447,AAACCATTCCAGCCCT,229.0,43.0,1872.0,S,2446,7.802618,10504,9.259607,...,2.833213,0.152323,0,0.000000,0.000000,1,5.438079,7.535297,3.784190,0.036716
AAACCATTCGTGACCG,2581,AAACCATTCGTGACCG,58.0,1.0,2.0,G1,2581,7.856320,8775,9.079776,...,3.367296,0.319088,0,0.000000,0.000000,1,4.077537,1.098612,0.693147,0.012962


In [14]:
%%time

### ----------------------------------------------
# Outputs from velocyto
### ----------------------------------------------

fpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/velocyto/scfib/merged_PW6SH.loom"

ldata = dyn.read_loom(
    fpath,
    sparse=True,
)

ldata.var_names_make_unique()

ldata.obs['velocyto_cell_id'] = ldata.obs_names.copy()
ldata.obs['cell_id'] = ldata.obs['velocyto_cell_id'].apply(lambda x: x.split(":")[1][:-1])
ldata.obs_names = ldata.obs['cell_id'].values

# Print initial cell counts
print(f"\tInitial cell count in ldata: {ldata.shape[0]}")
print(f"\tInitial cell count in adata: {adata.shape[0]}")

# IDs not retained in ldata
removed_from_ldata = ldata.obs.index[~ldata.obs.index.isin(adata.obs_names)].tolist()

# IDs not retained in adata
removed_from_adata = adata.obs.index[~adata.obs.index.isin(ldata.obs_names)].tolist()

# Filter ldata to keep only cells present in adata
initial_ldata_count = ldata.shape[0]
ldata = ldata[ldata.obs.index.isin(adata.obs_names), :].copy()
print(f"\tFiltered ldata: {ldata.shape[0]} cells kept, {initial_ldata_count - ldata.shape[0]} cells removed")

ldata

/nfs/turbo/umms-indikar/Jillian/conda-envs/dynamo/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


	Initial cell count in ldata: 8963
	Initial cell count in adata: 7748
	Filtered ldata: 7748 cells kept, 1215 cells removed
CPU times: user 30.8 s, sys: 11 s, total: 41.9 s
Wall time: 43.3 s


AnnData object with n_obs × n_vars = 7748 × 38606
    obs: 'velocyto_cell_id', 'cell_id'
    var: 'Accession', 'Chromosome', 'End', 'Start', 'Strand'
    layers: 'matrix', 'ambiguous', 'spliced', 'unspliced'

In [15]:
preprocessor = Preprocessor()
preprocessor.preprocess_adata(ldata, recipe="monocle")
ldata

|-----> Running monocle preprocessing pipeline...
|-----------> filtered out 8 outlier cells
|-----------> filtered out 32952 outlier genes
|-----> PCA dimension reduction
|-----> <insert> X_pca to obsm in AnnData Object.
|-----> [Preprocessor-monocle] completed [10.3253s]

╭─ SUMMARY: Preprocessor.preprocess_adata ───────────────────────────╮
│  Duration: 10.3299s                                                │
│  Shape:    7,748 x 38,606 -> 7,740 x 38,606                        │
│                                                                    │
│  CHANGES DETECTED                                                  │
│  ────────────────                                                  │
│  ● OBS    │ ✚ Size_Factor (float)                                  │
│           │ ✚ initial_cell_size (float)                            │
│           │ ✚ initial_spliced_cell_size (float)                    │
│           │ ✚ initial_unspliced_cell_size (float)                  │
│           │ ✚

AnnData object with n_obs × n_vars = 7740 × 38606
    obs: 'velocyto_cell_id', 'cell_id', 'nGenes', 'nCounts', 'pMito', 'pass_basic_filter', 'spliced_Size_Factor', 'initial_spliced_cell_size', 'Size_Factor', 'initial_cell_size', 'unspliced_Size_Factor', 'initial_unspliced_cell_size', 'ntr'
    var: 'Accession', 'Chromosome', 'End', 'Start', 'Strand', 'nCells', 'nCounts', 'pass_basic_filter', 'score', 'log_m', 'log_cv', 'frac', 'use_for_pca', 'ntr'
    uns: 'pp', 'velocyto_SVR', 'feature_selection', 'PCs', 'explained_variance_ratio_', 'pca_mean', 'history_log'
    obsm: 'X_pca'
    layers: 'matrix', 'ambiguous', 'spliced', 'unspliced', 'X_spliced', 'X_unspliced'

In [16]:
dyn.tl.dynamics(ldata)
ldata

|-----> dynamics_del_2nd_moments_key is None. Using default value from DynamoAdataConfig: dynamics_del_2nd_moments_key=False
|-----------> removing existing M layers:[]...
|-----------> making adata smooth...
|-----> calculating first/second moments...
|-----> [moments calculation] completed [53.4975s]


estimating gamma: 100%|██████████| 2000/2000 [02:59<00:00, 11.14it/s]



╭─ SUMMARY: dynamics ────────────────────────────────────────────────╮
│  Duration: 247.7801s                                               │
│  Shape:    7,740 x 38,606 (Unchanged)                              │
│                                                                    │
│  CHANGES DETECTED                                                  │
│  ────────────────                                                  │
│  ● VAR    │ ✚ use_for_dynamics (bool)                              │
│                                                                    │
│  ● UNS    │ ✚ dynamics                                             │
│           │ ✚ vel_params_names                                     │
│                                                                    │
│  ● OBSP   │ ✚ moments_con (sparse matrix, 7740x7740)               │
│                                                                    │
│  ● LAYERS │ ✚ M_s (sparse matrix, 7740x38606)                      │
│    

AnnData object with n_obs × n_vars = 7740 × 38606
    obs: 'velocyto_cell_id', 'cell_id', 'nGenes', 'nCounts', 'pMito', 'pass_basic_filter', 'spliced_Size_Factor', 'initial_spliced_cell_size', 'Size_Factor', 'initial_cell_size', 'unspliced_Size_Factor', 'initial_unspliced_cell_size', 'ntr'
    var: 'Accession', 'Chromosome', 'End', 'Start', 'Strand', 'nCells', 'nCounts', 'pass_basic_filter', 'score', 'log_m', 'log_cv', 'frac', 'use_for_pca', 'ntr', 'use_for_dynamics'
    uns: 'pp', 'velocyto_SVR', 'feature_selection', 'PCs', 'explained_variance_ratio_', 'pca_mean', 'history_log', 'vel_params_names', 'dynamics'
    obsm: 'X_pca'
    varm: 'vel_params'
    layers: 'matrix', 'ambiguous', 'spliced', 'unspliced', 'X_spliced', 'X_unspliced', 'M_u', 'M_uu', 'M_s', 'M_us', 'M_ss', 'velocity_S'
    obsp: 'moments_con'